# Benchmark Comparison: DP vs No-DP TT Methods

This notebook compares:
1. `yoni_laptop_no_dp_benchmark_tt.csv` - No DP benchmark
2. `yoni_laptop_dp_benchmark_tt.csv` - DP benchmark

## Analysis includes:
- Problems solved in yoni files but NOT in Ido's file
- Makespan validation between the 2 yoni files
- Problems where only DP finished vs only No-DP finished
- Expand number comparison when neither finished (with percentages)

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import os

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = [12, 6]
plt.rcParams['font.size'] = 11

# Data path
DATA_PATH = os.path.join(os.path.dirname(os.getcwd()), "data")
if not os.path.exists(DATA_PATH):
    DATA_PATH = "../data"

print(f"Data path: {DATA_PATH}")

## 1. Load Data

In [ ]:
# Load the three files
df_no_dp = pd.read_csv(os.path.join(DATA_PATH, "yoni_laptop_no_dp_benchmark_tt.csv"))
df_dp = pd.read_csv(os.path.join(DATA_PATH, "yoni_laptop_dp_benchmark_tt.csv"))
df_ido = pd.read_csv(os.path.join(DATA_PATH, "TT_ido.csv"))

# Normalize finished column
for df in [df_no_dp, df_dp]:
    if df['finished'].dtype == object:
        df['finished'] = df['finished'].str.upper() == 'TRUE'
    else:
        df['finished'] = df['finished'].astype(bool)

if df_ido['finished'].dtype == object:
    df_ido['finished'] = df_ido['finished'].str.upper() == 'TRUE'
else:
    df_ido['finished'] = df_ido['finished'].astype(bool)

print("Data loaded successfully!")
print(f"\nNo-DP: {len(df_no_dp)} rows, {df_no_dp['finished'].sum()} finished")
print(f"DP: {len(df_dp)} rows, {df_dp['finished'].sum()} finished")
print(f"Ido: {len(df_ido)} rows, {df_ido['finished'].sum()} finished")

## 2. Problems Solved in Yoni Files but NOT in Ido

In [ ]:
# Merge all three dataframes
merged = pd.merge(
    df_no_dp[['group', 'exam', 'finished', 'makespan', 'expand_number', 'time']],
    df_dp[['group', 'exam', 'finished', 'makespan', 'expand_number', 'time']],
    on=['group', 'exam'],
    suffixes=('_no_dp', '_dp')
)

# Add Ido's data
merged = pd.merge(
    merged,
    df_ido[['group', 'exam', 'finished', 'makespan']],
    on=['group', 'exam'],
    how='left'
)
merged = merged.rename(columns={'finished': 'finished_ido', 'makespan': 'makespan_ido'})

# Fill NaN for problems not in Ido's file
merged['finished_ido'] = merged['finished_ido'].fillna(False)

print(f"Total problems in merged dataset: {len(merged)}")

In [ ]:
# Problems solved in No-DP but NOT in Ido
no_dp_solved_ido_not = merged[(merged['finished_no_dp'] == True) & (merged['finished_ido'] == False)]

# Problems solved in DP but NOT in Ido
dp_solved_ido_not = merged[(merged['finished_dp'] == True) & (merged['finished_ido'] == False)]

# Problems solved in EITHER yoni file but NOT in Ido
either_solved_ido_not = merged[
    ((merged['finished_no_dp'] == True) | (merged['finished_dp'] == True)) & 
    (merged['finished_ido'] == False)
]

print("="*70)
print("PROBLEMS SOLVED IN YONI FILES BUT NOT IN IDO")
print("="*70)
print(f"\nNo-DP solved but Ido didn't: {len(no_dp_solved_ido_not)} problems")
print(f"DP solved but Ido didn't: {len(dp_solved_ido_not)} problems")
print(f"Either (No-DP or DP) solved but Ido didn't: {len(either_solved_ido_not)} problems")

In [ ]:
# Show details of problems solved in yoni files but not in Ido
if len(either_solved_ido_not) > 0:
    print("\nDetails of problems solved in Yoni files but NOT in Ido:")
    print("-"*70)
    
    display_df = either_solved_ido_not[['group', 'exam', 'finished_no_dp', 'finished_dp', 
                                         'makespan_no_dp', 'makespan_dp']].copy()
    display_df['solved_by'] = display_df.apply(
        lambda x: 'Both' if x['finished_no_dp'] and x['finished_dp'] 
                  else ('No-DP only' if x['finished_no_dp'] else 'DP only'), axis=1
    )
    display_df = display_df.sort_values(['group', 'exam'])
    display(display_df)
    
    # Summary by who solved
    print("\nBreakdown:")
    print(display_df['solved_by'].value_counts())

## 3. Makespan Validation Between the 2 Yoni Files

In [ ]:
# Filter for problems finished in BOTH yoni files
both_yoni_finished = merged[
    (merged['finished_no_dp'] == True) & 
    (merged['finished_dp'] == True)
].copy()

# Check makespan equality
both_yoni_finished['makespan_match'] = both_yoni_finished['makespan_no_dp'] == both_yoni_finished['makespan_dp']

print("="*70)
print("MAKESPAN VALIDATION (Between the 2 Yoni files)")
print("="*70)
print(f"\nProblems finished in BOTH No-DP and DP: {len(both_yoni_finished)}")
print(f"Makespans matching: {both_yoni_finished['makespan_match'].sum()}")
print(f"Makespans NOT matching: {(~both_yoni_finished['makespan_match']).sum()}")

if both_yoni_finished['makespan_match'].all():
    print("\n*** SUCCESS: All makespans match between No-DP and DP! ***")
else:
    print("\n*** WARNING: Some makespans do not match! ***")
    mismatches = both_yoni_finished[~both_yoni_finished['makespan_match']]
    display(mismatches[['group', 'exam', 'makespan_no_dp', 'makespan_dp']])

## 4. Problems: DP Solved vs No-DP Solved

In [ ]:
# Create categories
merged['category'] = 'Unknown'
merged.loc[(merged['finished_no_dp'] == True) & (merged['finished_dp'] == True), 'category'] = 'Both Finished'
merged.loc[(merged['finished_no_dp'] == True) & (merged['finished_dp'] == False), 'category'] = 'Only No-DP Finished'
merged.loc[(merged['finished_no_dp'] == False) & (merged['finished_dp'] == True), 'category'] = 'Only DP Finished'
merged.loc[(merged['finished_no_dp'] == False) & (merged['finished_dp'] == False), 'category'] = 'Neither Finished'

category_counts = merged['category'].value_counts()
total = len(merged)

print("="*70)
print("PROBLEM CATEGORIES: DP vs No-DP")
print("="*70)
print(f"\nTotal problems: {total}")
print("\nCategory breakdown:")
for cat in ['Both Finished', 'Only DP Finished', 'Only No-DP Finished', 'Neither Finished']:
    count = category_counts.get(cat, 0)
    pct = count / total * 100
    print(f"  {cat}: {count} ({pct:.1f}%)")

In [ ]:
# Visualization: Problem categories
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Pie chart
colors = ['#2ecc71', '#3498db', '#e74c3c', '#95a5a6']
category_order = ['Both Finished', 'Only DP Finished', 'Only No-DP Finished', 'Neither Finished']
counts = [category_counts.get(cat, 0) for cat in category_order]

wedges, texts, autotexts = axes[0].pie(
    counts, 
    labels=category_order,
    autopct='%1.1f%%',
    colors=colors,
    explode=(0.02, 0.02, 0.02, 0.02)
)
axes[0].set_title('Problem Categories: DP vs No-DP', fontsize=14, fontweight='bold')

# Bar chart
x = np.arange(len(category_order))
bars = axes[1].bar(x, counts, color=colors, edgecolor='black', linewidth=1.2)
axes[1].set_xticks(x)
axes[1].set_xticklabels(category_order, rotation=15, ha='right')
axes[1].set_ylabel('Number of Problems', fontsize=12)
axes[1].set_title('Problem Categories Distribution', fontsize=14, fontweight='bold')

# Add value labels on bars
for bar, count in zip(bars, counts):
    pct = count / total * 100
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 2, 
                 f'{count}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold')

plt.tight_layout()
plt.show()

In [ ]:
# Show problems where ONLY DP finished
only_dp = merged[merged['category'] == 'Only DP Finished'].copy()

print("="*70)
print(f"PROBLEMS WHERE ONLY DP FINISHED ({len(only_dp)} problems)")
print("="*70)

if len(only_dp) > 0:
    display(only_dp[['group', 'exam', 'makespan_dp', 'time_dp', 'expand_number_dp', 
                     'time_no_dp', 'expand_number_no_dp']].sort_values(['group', 'exam']))
else:
    print("No problems where only DP finished.")

In [ ]:
# Show problems where ONLY No-DP finished
only_no_dp = merged[merged['category'] == 'Only No-DP Finished'].copy()

print("="*70)
print(f"PROBLEMS WHERE ONLY No-DP FINISHED ({len(only_no_dp)} problems)")
print("="*70)

if len(only_no_dp) > 0:
    display(only_no_dp[['group', 'exam', 'makespan_no_dp', 'time_no_dp', 'expand_number_no_dp',
                        'time_dp', 'expand_number_dp']].sort_values(['group', 'exam']))
else:
    print("No problems where only No-DP finished.")

## 5. Expand Number Analysis: Neither Finished

In [ ]:
# Problems not finished in both
neither_finished = merged[merged['category'] == 'Neither Finished'].copy()

print("="*70)
print("EXPAND NUMBER ANALYSIS (Problems NOT finished in BOTH)")
print("="*70)
print(f"\nTotal problems not finished in both: {len(neither_finished)}")

if len(neither_finished) > 0:
    # Compare expand numbers
    neither_finished['expand_diff'] = neither_finished['expand_number_dp'] - neither_finished['expand_number_no_dp']
    neither_finished['expand_ratio'] = neither_finished['expand_number_dp'] / neither_finished['expand_number_no_dp']
    neither_finished['dp_expanded_more'] = neither_finished['expand_number_dp'] > neither_finished['expand_number_no_dp']
    neither_finished['no_dp_expanded_more'] = neither_finished['expand_number_no_dp'] > neither_finished['expand_number_dp']
    neither_finished['equal_expand'] = neither_finished['expand_number_dp'] == neither_finished['expand_number_no_dp']
    
    dp_more = neither_finished['dp_expanded_more'].sum()
    no_dp_more = neither_finished['no_dp_expanded_more'].sum()
    equal = neither_finished['equal_expand'].sum()
    total_neither = len(neither_finished)
    
    print(f"\n" + "-"*50)
    print("WHO EXPANDED MORE NODES? (Statistics in Percent)")
    print("-"*50)
    print(f"  DP expanded more:    {dp_more:3d} problems ({dp_more/total_neither*100:5.1f}%)")
    print(f"  No-DP expanded more: {no_dp_more:3d} problems ({no_dp_more/total_neither*100:5.1f}%)")
    print(f"  Equal:               {equal:3d} problems ({equal/total_neither*100:5.1f}%)")
    
    print(f"\n" + "-"*50)
    print("EXPAND NUMBER STATISTICS")
    print("-"*50)
    print(f"  DP - Mean:   {neither_finished['expand_number_dp'].mean():>12,.0f}")
    print(f"  DP - Median: {neither_finished['expand_number_dp'].median():>12,.0f}")
    print(f"  DP - Min:    {neither_finished['expand_number_dp'].min():>12,.0f}")
    print(f"  DP - Max:    {neither_finished['expand_number_dp'].max():>12,.0f}")
    print()
    print(f"  No-DP - Mean:   {neither_finished['expand_number_no_dp'].mean():>12,.0f}")
    print(f"  No-DP - Median: {neither_finished['expand_number_no_dp'].median():>12,.0f}")
    print(f"  No-DP - Min:    {neither_finished['expand_number_no_dp'].min():>12,.0f}")
    print(f"  No-DP - Max:    {neither_finished['expand_number_no_dp'].max():>12,.0f}")
    
    print(f"\n" + "-"*50)
    print("RATIO STATISTICS (DP / No-DP)")
    print("-"*50)
    print(f"  Mean ratio:   {neither_finished['expand_ratio'].mean():.3f}")
    print(f"  Median ratio: {neither_finished['expand_ratio'].median():.3f}")
    print(f"  Min ratio:    {neither_finished['expand_ratio'].min():.3f}")
    print(f"  Max ratio:    {neither_finished['expand_ratio'].max():.3f}")
else:
    print("\nNo problems where neither finished!")

In [ ]:
# Visualization: Expand number comparison for neither finished
if len(neither_finished) > 0:
    fig, axes = plt.subplots(2, 2, figsize=(14, 12))
    
    # 1. Bar chart: Who expanded more (with percentages)
    ax1 = axes[0, 0]
    categories = ['DP Expanded More', 'No-DP Expanded More', 'Equal']
    values = [dp_more, no_dp_more, equal]
    colors = ['#3498db', '#e74c3c', '#95a5a6']
    bars = ax1.bar(categories, values, color=colors, edgecolor='black', linewidth=1.2)
    ax1.set_ylabel('Number of Problems', fontsize=12)
    ax1.set_title('Who Expanded More Nodes?\n(Neither Finished)', fontsize=14, fontweight='bold')
    for bar, val in zip(bars, values):
        pct = val / total_neither * 100
        ax1.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5, 
                 f'{val}\n({pct:.1f}%)', ha='center', va='bottom', fontweight='bold', fontsize=11)
    
    # 2. Scatter plot: DP vs No-DP expand numbers
    ax2 = axes[0, 1]
    ax2.scatter(neither_finished['expand_number_no_dp'], neither_finished['expand_number_dp'], 
                alpha=0.6, c='#e74c3c', edgecolors='black', linewidth=0.5, s=60)
    max_expand = max(neither_finished['expand_number_no_dp'].max(), neither_finished['expand_number_dp'].max())
    ax2.plot([0, max_expand], [0, max_expand], 'k--', label='Equal line', linewidth=2)
    ax2.set_xlabel('No-DP Expand Number', fontsize=12)
    ax2.set_ylabel('DP Expand Number', fontsize=12)
    ax2.set_title('Expand Number: DP vs No-DP\n(Neither Finished)', fontsize=14, fontweight='bold')
    ax2.legend()
    ax2.ticklabel_format(style='scientific', axis='both', scilimits=(0,0))
    
    # 3. Histogram of expand ratio
    ax3 = axes[1, 0]
    ax3.hist(neither_finished['expand_ratio'], bins=20, color='#9b59b6', edgecolor='black', alpha=0.7)
    ax3.axvline(x=1, color='red', linestyle='--', linewidth=2, label='Ratio = 1 (Equal)')
    ax3.axvline(x=neither_finished['expand_ratio'].mean(), color='green', linestyle='-', linewidth=2, 
                label=f'Mean = {neither_finished["expand_ratio"].mean():.3f}')
    ax3.set_xlabel('Expand Ratio (DP / No-DP)', fontsize=12)
    ax3.set_ylabel('Frequency', fontsize=12)
    ax3.set_title('Distribution of Expand Ratio\n(Neither Finished)', fontsize=14, fontweight='bold')
    ax3.legend()
    
    # 4. Box plot comparison
    ax4 = axes[1, 1]
    data_to_plot = [neither_finished['expand_number_no_dp'], neither_finished['expand_number_dp']]
    bp = ax4.boxplot(data_to_plot, labels=['No-DP', 'DP'], patch_artist=True)
    bp['boxes'][0].set_facecolor('#e74c3c')
    bp['boxes'][1].set_facecolor('#3498db')
    ax4.set_ylabel('Expand Number', fontsize=12)
    ax4.set_title('Expand Number Distribution\n(Neither Finished)', fontsize=14, fontweight='bold')
    ax4.ticklabel_format(style='scientific', axis='y', scilimits=(0,0))
    
    plt.tight_layout()
    plt.show()

In [ ]:
# Show detailed table of problems not finished in both
if len(neither_finished) > 0:
    print("\nDetailed list of problems NOT finished in both:")
    print("-"*70)
    
    detail_df = neither_finished[['group', 'exam', 'expand_number_no_dp', 'expand_number_dp', 
                                   'expand_diff', 'expand_ratio']].copy()
    detail_df['who_expanded_more'] = neither_finished.apply(
        lambda x: 'DP' if x['dp_expanded_more'] else ('No-DP' if x['no_dp_expanded_more'] else 'Equal'), axis=1
    )
    detail_df = detail_df.sort_values(['group', 'exam'])
    detail_df['expand_ratio'] = detail_df['expand_ratio'].round(3)
    
    display(detail_df)

## 6. Summary

In [ ]:
# Final summary
print("="*70)
print("FINAL SUMMARY")
print("="*70)

print(f"\n1. PROBLEMS SOLVED IN YONI BUT NOT IN IDO:")
print(f"   - Either No-DP or DP solved: {len(either_solved_ido_not)} problems")

print(f"\n2. MAKESPAN VALIDATION (Between No-DP and DP):")
if len(both_yoni_finished) > 0:
    match_pct = both_yoni_finished['makespan_match'].mean() * 100
    print(f"   - {len(both_yoni_finished)} problems finished in both")
    print(f"   - {both_yoni_finished['makespan_match'].sum()} makespans match ({match_pct:.1f}%)")
else:
    print("   - No problems finished in both")

print(f"\n3. PROBLEM CATEGORIES:")
for cat in ['Both Finished', 'Only DP Finished', 'Only No-DP Finished', 'Neither Finished']:
    count = category_counts.get(cat, 0)
    pct = count / total * 100
    print(f"   - {cat}: {count} ({pct:.1f}%)")

print(f"\n4. EXPAND NUMBER (When Neither Finished):")
if len(neither_finished) > 0:
    print(f"   - Total: {len(neither_finished)} problems")
    print(f"   - DP expanded more: {dp_more} ({dp_more/total_neither*100:.1f}%)")
    print(f"   - No-DP expanded more: {no_dp_more} ({no_dp_more/total_neither*100:.1f}%)")
    print(f"   - Equal: {equal} ({equal/total_neither*100:.1f}%)")
    print(f"   - Average ratio (DP/No-DP): {neither_finished['expand_ratio'].mean():.3f}")
else:
    print("   - No problems where neither finished")

In [ ]:
# Final visualization
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# 1. Categories pie
ax1 = axes[0]
colors = ['#2ecc71', '#3498db', '#e74c3c', '#95a5a6']
category_order = ['Both Finished', 'Only DP Finished', 'Only No-DP Finished', 'Neither Finished']
counts = [category_counts.get(cat, 0) for cat in category_order]
ax1.pie(counts, labels=category_order, autopct='%1.1f%%', colors=colors, explode=(0.02, 0.02, 0.02, 0.02))
ax1.set_title('Problem Categories', fontsize=14, fontweight='bold')

# 2. Who expanded more (neither finished)
ax2 = axes[1]
if len(neither_finished) > 0:
    cats = ['DP More', 'No-DP More', 'Equal']
    vals = [dp_more, no_dp_more, equal]
    colors2 = ['#3498db', '#e74c3c', '#95a5a6']
    bars = ax2.bar(cats, vals, color=colors2, edgecolor='black')
    for bar, val in zip(bars, vals):
        pct = val / total_neither * 100
        ax2.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3, 
                 f'{pct:.1f}%', ha='center', fontweight='bold')
ax2.set_ylabel('Count')
ax2.set_title('Who Expanded More?\n(Neither Finished)', fontsize=14, fontweight='bold')

# 3. DP vs No-DP advantage
ax3 = axes[2]
only_dp_count = category_counts.get('Only DP Finished', 0)
only_no_dp_count = category_counts.get('Only No-DP Finished', 0)
ax3.bar(['Only DP Finished', 'Only No-DP Finished'], [only_dp_count, only_no_dp_count], 
        color=['#3498db', '#e74c3c'], edgecolor='black')
for i, val in enumerate([only_dp_count, only_no_dp_count]):
    pct = val / total * 100
    ax3.text(i, val + 0.3, f'{val}\n({pct:.1f}%)', ha='center', fontweight='bold')
ax3.set_ylabel('Count')
ax3.set_title('Exclusive Solves', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.show()